In [4]:
#!/usr/bin/env python3
"""
compare_energy_uncertainty.py

Compares:
  (A) OLD benchmark GMFD vs NEW GMFD:
      - coefficient CSVs (parm,beta)
      - panel .dta datasets (key alignment + variable comparisons)
  (B) NEW vs NEW (optional): e.g. GMFD vs ERA5 panel comparison

Outputs:
  - CSVs of coefficient diffs
  - CSVs of panel variable diffs (summary + worst offenders)
  - small text summary to stdout

Assumptions:
  - OLD coeff CSVs live in:  .../output_original/sters
  - NEW coeff CSVs live in:  .../data/regression/sters
  - NEW panel .dta live in:  .../data/regression
  - You can point OLD panel .dta explicitly if stored elsewhere.

Usage (example):
  python compare_energy_uncertainty.py \
    --old-ster-dir /user/ab5405/summeraliaclimate/code/energy_uncertainty/output_original/sters \
    --new-ster-dir /user/ab5405/summeraliaclimate/code/energy_uncertainty/data/regression/sters \
    --new-regd /user/ab5405/summeraliaclimate/code/energy_uncertainty/data/regression \
    --product GMFD \
    --old-panel /path/to/OLD/GMFD_TINV_clim_regsort.dta \
    --new-panel /user/ab5405/summeraliaclimate/code/energy_uncertainty/data/regression/GMFD_TINV_clim_regsort.dta
"""

import sys

if "ipykernel" in sys.argv[0]:
    sys.argv = [
        sys.argv[0],
        "--old-ster-dir", "/user/ab5405/summeraliaclimate/code/energy_uncertainty/output_original/sters",
        "--new-ster-dir", "/user/ab5405/summeraliaclimate/code/energy_uncertainty/data/regression/sters",
        "--new-regd", "/user/ab5405/summeraliaclimate/code/energy_uncertainty/data/regression",
        "--product", "GMFD",
        "--old-panel", "/user/ab5405/summeraliaclimate/code/energy_consumption/energy_data_release_2021oct21/DATA/regression/GMFD_TINV_clim_regsort.dta",
        "--new-panel", "/user/ab5405/summeraliaclimate/code/energy_uncertainty/data/regression/GMFD_TINV_clim_regsort.dta",
    ]

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd


# -----------------------------
# Utility
# -----------------------------

def _safe_read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    return pd.read_csv(path)

def _safe_read_dta(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    # pandas can read Stata .dta
    return pd.read_stata(path, convert_categoricals=False)

def _mkdir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)

def _is_num(s: pd.Series) -> bool:
    return pd.api.types.is_numeric_dtype(s)

def _fmt(x) -> str:
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "NA"
    if isinstance(x, (int, np.integer)):
        return str(int(x))
    if isinstance(x, (float, np.floating)):
        return f"{x:.6g}"
    return str(x)

def _relative_diff(new: pd.Series, old: pd.Series) -> pd.Series:
    # robust relative diff: (new-old)/max(|old|, eps)
    eps = 1e-12
    denom = np.maximum(np.abs(old), eps)
    return (new - old) / denom


# -----------------------------
# Coefficient comparison
# -----------------------------

def compare_coeff_csv(old_csv: Path, new_csv: Path, out_csv: Path) -> pd.DataFrame:
    """
    Expects columns:
      old: parm, beta
      new: parm, beta
    """
    old = _safe_read_csv(old_csv)
    new = _safe_read_csv(new_csv)

    for df, label in [(old, "old"), (new, "new")]:
        if not {"parm", "beta"}.issubset(df.columns):
            raise ValueError(f"{label} coeff CSV missing columns parm,beta: {df.columns.tolist()}")

    comp = old.merge(new, on="parm", how="outer", suffixes=("_old", "_new"), indicator=True)
    comp["diff"] = comp["beta_new"] - comp["beta_old"]
    comp["rel_diff"] = _relative_diff(comp["beta_new"], comp["beta_old"])
    comp["abs_diff"] = comp["diff"].abs()
    comp["rel_diff_pct"] = 100.0 * comp["rel_diff"]

    # Useful sorting helpers
    comp["abs_old"] = comp["beta_old"].abs()
    comp["abs_new"] = comp["beta_new"].abs()

    _mkdir(out_csv.parent)
    comp.sort_values(["_merge", "abs_diff"], ascending=[True, False]).to_csv(out_csv, index=False)

    return comp


# -----------------------------
# Panel comparisons
# -----------------------------

def check_keys(df: pd.DataFrame, keys: list[str], label: str) -> None:
    missing = [k for k in keys if k not in df.columns]
    if missing:
        raise ValueError(f"{label}: missing key columns {missing}. Have: {df.columns.tolist()[:30]}...")

    dup = df.duplicated(keys).sum()
    if dup > 0:
        raise ValueError(f"{label}: {dup} duplicate rows on keys={keys}")

def align_on_keys(old: pd.DataFrame, new: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    merged = old.merge(new, on=keys, how="outer", suffixes=("_old", "_new"), indicator=True)
    return merged

def summarize_merge(merged: pd.DataFrame) -> dict:
    counts = merged["_merge"].value_counts(dropna=False).to_dict()
    return {
        "n_rows": int(len(merged)),
        "both": int(counts.get("both", 0)),
        "left_only": int(counts.get("left_only", 0)),
        "right_only": int(counts.get("right_only", 0)),
    }

def compare_numeric_column(merged: pd.DataFrame, col: str) -> dict:
    o = merged[f"{col}_old"]
    n = merged[f"{col}_new"]

    # only compare where both present
    m = merged["_merge"] == "both"
    o2 = o[m]
    n2 = n[m]

    out = {
        "col": col,
        "n_both": int(m.sum()),
        "old_mean": float(np.nanmean(o2)) if len(o2) else np.nan,
        "new_mean": float(np.nanmean(n2)) if len(n2) else np.nan,
        "old_sd": float(np.nanstd(o2)) if len(o2) else np.nan,
        "new_sd": float(np.nanstd(n2)) if len(n2) else np.nan,
        "corr": float(o2.corr(n2)) if (len(o2) and o2.notna().any() and n2.notna().any()) else np.nan,
    }

    diff = (n2 - o2)
    out["mean_diff"] = float(np.nanmean(diff)) if len(diff) else np.nan
    out["mean_abs_diff"] = float(np.nanmean(np.abs(diff))) if len(diff) else np.nan

    # robust relative diff using old magnitude
    rel = _relative_diff(n2, o2)
    out["mean_abs_rel_diff_pct"] = float(np.nanmean(np.abs(100.0 * rel))) if len(rel) else np.nan

    # percentile diagnostics
    for q in [50, 90, 95, 99]:
        out[f"p{q}_abs_diff"] = float(np.nanpercentile(np.abs(diff), q)) if len(diff) else np.nan
        out[f"p{q}_abs_rel_diff_pct"] = float(np.nanpercentile(np.abs(100.0 * rel), q)) if len(rel) else np.nan

    return out

def panel_variable_report(
    old_panel: Path,
    new_panel: Path,
    out_dir: Path,
    keys: list[str],
    include_patterns: list[str] | None = None,
    exclude_patterns: list[str] | None = None,
    name: str = "panel_compare",
) -> None:
    old = _safe_read_dta(old_panel)
    new = _safe_read_dta(new_panel)

    check_keys(old, keys, "OLD panel")
    check_keys(new, keys, "NEW panel")

    merged = align_on_keys(old, new, keys)

    # merge summary
    summ = summarize_merge(merged)
    print(f"\n[{name}] key={keys} rows={summ['n_rows']} both={summ['both']} "
          f"left_only={summ['left_only']} right_only={summ['right_only']}")

    _mkdir(out_dir)

    # choose comparable columns
    old_cols = set(old.columns) - set(keys)
    new_cols = set(new.columns) - set(keys)
    common = sorted(list(old_cols & new_cols))

    def _match_any(col: str, pats: list[str]) -> bool:
        return any(p in col for p in pats)

    if include_patterns:
        common = [c for c in common if _match_any(c, include_patterns)]
    if exclude_patterns:
        common = [c for c in common if not _match_any(c, exclude_patterns)]

    # numeric-only comparisons
    numeric_common = []
    for c in common:
        if _is_num(old[c]) and _is_num(new[c]):
            numeric_common.append(c)

    if not numeric_common:
        raise RuntimeError(f"[{name}] No numeric columns to compare after filters.")

    rows = []
    for c in numeric_common:
        rows.append(compare_numeric_column(merged, c))

    rep = pd.DataFrame(rows)

    # Save full report
    rep_path = out_dir / f"{name}_var_summary.csv"
    rep.sort_values(["mean_abs_diff"], ascending=False).to_csv(rep_path, index=False)

    # Save top “worst” variables
    worst_path = out_dir / f"{name}_worst20_by_absdiff.csv"
    rep.sort_values(["mean_abs_diff"], ascending=False).head(20).to_csv(worst_path, index=False)

    # Also: for the worst variable, dump row-level diffs (helpful for debugging)
    worst_col = rep.sort_values(["mean_abs_diff"], ascending=False).iloc[0]["col"]
    m = merged["_merge"] == "both"
    dd = merged.loc[m, keys + [f"{worst_col}_old", f"{worst_col}_new"]].copy()
    dd["diff"] = dd[f"{worst_col}_new"] - dd[f"{worst_col}_old"]
    dd["abs_diff"] = dd["diff"].abs()
    dd.sort_values("abs_diff", ascending=False).head(200).to_csv(out_dir / f"{name}_worstcol_rows.csv", index=False)

    print(f"[{name}] wrote:")
    print(f"  - {rep_path}")
    print(f"  - {worst_path}")
    print(f"  - {out_dir / f'{name}_worstcol_rows.csv'}")


def add_fd_vars(df: pd.DataFrame, keys: list[str], base_cols: list[str], group_key: str) -> pd.DataFrame:
    """
    Adds FD_{col} = within-group first difference for each col in base_cols.
    Assumes df is at region-year or iso-year resolution.
    """
    df = df.sort_values(keys).copy()
    for c in base_cols:
        if c in df.columns:
            df[f"FD_{c}"] = df.groupby(group_key)[c].diff()
    return df


# -----------------------------
# Main
# -----------------------------

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--old-ster-dir", type=Path, required=True)
    ap.add_argument("--new-ster-dir", type=Path, required=True)
    ap.add_argument("--new-regd", type=Path, required=True)

    ap.add_argument("--product", type=str, default="GMFD")
    ap.add_argument("--model-name", type=str, default="TINV_clim")  # or "TINV_clim_quadinter", etc.

    ap.add_argument("--old-panel", type=Path, required=True, help="Path to OLD benchmark GMFD panel .dta")
    ap.add_argument("--new-panel", type=Path, required=False, help="Path to NEW GMFD panel .dta (defaults to new-regd/{PRODUCT}_TINV_clim_regsort.dta)")

    ap.add_argument("--keys", type=str, default="region_i,year", help="Comma-separated key columns")
    ap.add_argument("--group-key", type=str, default="region_i", help="Grouping key for FD diagnostics")

    ap.add_argument("--compare-new-product", type=str, default=None, help="Optional: compare NEW GMFD panel vs NEW <product> panel")
    ap.add_argument("--new-product-panel", type=Path, default=None, help="Optional explicit path to NEW other product panel .dta")

    args = ap.parse_args()

    product = args.product
    keys = [k.strip() for k in args.keys.split(",") if k.strip()]

    old_dir = args.old_ster_dir
    new_dir = args.new_ster_dir
    out_dir = new_dir / "comparisons"
    _mkdir(out_dir)

    # -----------------------------
    # 1) Coefficient comparisons (OLD vs NEW GMFD)
    # -----------------------------
    # Customize these to match your naming scheme.
    # OLD benchmark files appear NOT product-suffixed; NEW are product-suffixed.
    coeff_pairs = [
        # global FGLS
        (old_dir / "FD_FGLS_global_TINV_clim_coeff.csv",
         new_dir / f"FD_FGLS_global_{product}_coeff.csv",
         out_dir / f"FD_FGLS_global_{product}_coeff_comparison.csv"),
        # inter FGLS quadinter
        (old_dir / "FD_FGLS_inter_TINV_clim_quadinter_coeff.csv",
         new_dir / f"FD_FGLS_inter_TINV_clim_quadinter_{product}_coeff.csv",
         out_dir / f"FD_FGLS_inter_TINV_clim_quadinter_{product}_coeff_comparison.csv"),
    ]

    print("\n=== COEFFICIENT COMPARISONS (OLD vs NEW) ===")
    for old_csv, new_csv, out_csv in coeff_pairs:
        print(f"\n[coeff] OLD: {old_csv}")
        print(f"[coeff] NEW: {new_csv}")
        comp = compare_coeff_csv(old_csv, new_csv, out_csv)
        both = comp[comp["_merge"] == "both"]
        print(f"[coeff] overlapping parms: {len(both)}")
        if len(both):
            print(f"[coeff] max |diff|: {_fmt(both['abs_diff'].max())}")
            print(f"[coeff] max |rel diff| (%): {_fmt((both['rel_diff_pct'].abs()).max())}")
            print(f"[coeff] mean |diff|: {_fmt(both['abs_diff'].mean())}")
        print(f"[coeff] wrote: {out_csv}")

    # -----------------------------
    # 2) Panel data comparison (OLD vs NEW GMFD)
    # -----------------------------
    print("\n=== PANEL COMPARISON (OLD vs NEW GMFD) ===")
    new_panel = args.new_panel
    if new_panel is None:
        new_panel = args.new_regd / f"{product}_TINV_clim_regsort.dta"

    # Compare key climate level vars + FD vars + key outcome
    # You can widen this list later; start with the “sanity core”.
    core_vars = [
        "load_pc",
        "precip1", "precip2",
        "temp1", "temp2", "temp3", "temp4",
        "hdd20", "cdd20",
    ]

    # For your panels, the precip/temp vars may be named FD_precip1 etc, or precip1 with suffix earlier.
    # We will compare whatever overlaps in BOTH panels; and we’ll also compute FD diagnostics for core_vars if present.
    old_df = _safe_read_dta(args.old_panel)
    new_df = _safe_read_dta(new_panel)

    check_keys(old_df, keys, "OLD panel")
    check_keys(new_df, keys, "NEW panel")

    # Add FD columns for any core vars present in BOTH
    group_key = args.group_key
    old_df2 = add_fd_vars(old_df, keys, core_vars, group_key)
    new_df2 = add_fd_vars(new_df, keys, core_vars, group_key)

    # Write temp copies to disk? not needed; do comparison in-memory by reusing report function
    # We'll save to temporary parquet? keep simple: just run report by merging ourselves.
    merged = align_on_keys(old_df2, new_df2, keys)
    summ = summarize_merge(merged)
    print(f"[panel] rows={summ['n_rows']} both={summ['both']} left_only={summ['left_only']} right_only={summ['right_only']}")

    # Decide which columns to focus on:
    # - All numeric overlapping columns by default, but exclude very high-dimensional FE artifacts if present
    exclude = ["__", "e(sample)"]  # add patterns if you want

    # Build a report on ALL numeric common columns
    # (This is the “panel lines up” check.)
    # To reuse panel_variable_report, we’ll write the modified dfs out as temporary stata? too annoying.
    # Instead: do the same computation inline.
    old_cols = set(old_df2.columns) - set(keys)
    new_cols = set(new_df2.columns) - set(keys)
    common = sorted(list(old_cols & new_cols))
    common = [c for c in common if not any(p in c for p in exclude)]
    num_common = [c for c in common if _is_num(old_df2[c]) and _is_num(new_df2[c])]

    rows = [compare_numeric_column(merged, c) for c in num_common]
    rep = pd.DataFrame(rows)

    rep_path = out_dir / f"panel_{product}_old_vs_new_var_summary.csv"
    rep.sort_values("mean_abs_diff", ascending=False).to_csv(rep_path, index=False)

    worst_path = out_dir / f"panel_{product}_old_vs_new_worst20_by_absdiff.csv"
    rep.sort_values("mean_abs_diff", ascending=False).head(20).to_csv(worst_path, index=False)

    # Special: precip/hdd/cdd focus (levels + FD_)
    focus = [c for c in rep["col"].tolist() if any(x in c for x in ["precip", "hdd", "cdd", "FD_precip", "FD_hdd", "FD_cdd"])]
    focus_rep = rep[rep["col"].isin(focus)].sort_values("mean_abs_diff", ascending=False)
    focus_path = out_dir / f"panel_{product}_old_vs_new_precip_hdd_cdd_focus.csv"
    focus_rep.to_csv(focus_path, index=False)

    print("[panel] wrote:")
    print(f"  - {rep_path}")
    print(f"  - {worst_path}")
    print(f"  - {focus_path}")

    # Dump top 200 row-level diffs for precip2 if present (this is often the canary)
    for target in ["precip1", "precip2", "FD_precip1", "FD_precip2"]:
        c_old = f"{target}_old"
        c_new = f"{target}_new"
        if c_old in merged.columns and c_new in merged.columns:
            dd = merged.loc[merged["_merge"] == "both", keys + [c_old, c_new]].copy()
            dd["diff"] = dd[c_new] - dd[c_old]
            dd["abs_diff"] = dd["diff"].abs()
            dd.sort_values("abs_diff", ascending=False).head(200).to_csv(
                out_dir / f"panel_{product}_old_vs_new_{target}_worst_rows.csv",
                index=False
            )
            print(f"[panel] wrote row-level worst diffs for {target}")

    # -----------------------------
    # 3) Optional NEW vs NEW panel comparison (GMFD vs another product)
    # -----------------------------
    if args.compare_new_product:
        other = args.compare_new_product
        other_panel = args.new_product_panel
        if other_panel is None:
            other_panel = args.new_regd / f"{other}_TINV_clim_regsort.dta"

        print(f"\n=== PANEL COMPARISON (NEW {product} vs NEW {other}) ===")
        # same procedure: compute FD diagnostics for core vars
        gmfd_df = new_df
        oth_df = _safe_read_dta(other_panel)
        check_keys(gmfd_df, keys, f"NEW {product} panel")
        check_keys(oth_df, keys, f"NEW {other} panel")

        gmfd_df2 = add_fd_vars(gmfd_df, keys, core_vars, group_key)
        oth_df2 = add_fd_vars(oth_df, keys, core_vars, group_key)

        merged2 = align_on_keys(gmfd_df2, oth_df2, keys)
        summ2 = summarize_merge(merged2)
        print(f"[panel new-new] rows={summ2['n_rows']} both={summ2['both']} left_only={summ2['left_only']} right_only={summ2['right_only']}")

        gmfd_cols = set(gmfd_df2.columns) - set(keys)
        oth_cols = set(oth_df2.columns) - set(keys)
        common2 = sorted(list(gmfd_cols & oth_cols))
        common2 = [c for c in common2 if not any(p in c for p in exclude)]
        num_common2 = [c for c in common2 if _is_num(gmfd_df2[c]) and _is_num(oth_df2[c])]

        rows2 = [compare_numeric_column(merged2, c) for c in num_common2]
        rep2 = pd.DataFrame(rows2)

        rep2_path = out_dir / f"panel_new_{product}_vs_{other}_var_summary.csv"
        rep2.sort_values("mean_abs_diff", ascending=False).to_csv(rep2_path, index=False)

        focus2 = [c for c in rep2["col"].tolist() if any(x in c for x in ["precip", "hdd", "cdd", "FD_precip", "FD_hdd", "FD_cdd"])]
        rep2_focus_path = out_dir / f"panel_new_{product}_vs_{other}_precip_hdd_cdd_focus.csv"
        rep2[rep2["col"].isin(focus2)].sort_values("mean_abs_diff", ascending=False).to_csv(rep2_focus_path, index=False)

        print("[panel new-new] wrote:")
        print(f"  - {rep2_path}")
        print(f"  - {rep2_focus_path}")


if __name__ == "__main__":
    main()



=== COEFFICIENT COMPARISONS (OLD vs NEW) ===

[coeff] OLD: /user/ab5405/summeraliaclimate/code/energy_uncertainty/output_original/sters/FD_FGLS_global_TINV_clim_coeff.csv
[coeff] NEW: /user/ab5405/summeraliaclimate/code/energy_uncertainty/data/regression/sters/FD_FGLS_global_GMFD_coeff.csv
[coeff] overlapping parms: 1
[coeff] max |diff|: 0.00616741
[coeff] max |rel diff| (%): 11.7747
[coeff] mean |diff|: 0.00616741
[coeff] wrote: /user/ab5405/summeraliaclimate/code/energy_uncertainty/data/regression/sters/comparisons/FD_FGLS_global_GMFD_coeff_comparison.csv

[coeff] OLD: /user/ab5405/summeraliaclimate/code/energy_uncertainty/output_original/sters/FD_FGLS_inter_TINV_clim_quadinter_coeff.csv
[coeff] NEW: /user/ab5405/summeraliaclimate/code/energy_uncertainty/data/regression/sters/FD_FGLS_inter_TINV_clim_quadinter_GMFD_coeff.csv
[coeff] overlapping parms: 33
[coeff] max |diff|: 13.7432
[coeff] max |rel diff| (%): 1102.22
[coeff] mean |diff|: 0.653139
[coeff] wrote: /user/ab5405/summerali

/user/ab5405/.conda/envs/hle_iv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/user/ab5405/.conda/envs/hle_iv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]


[panel] wrote:
  - /user/ab5405/summeraliaclimate/code/energy_uncertainty/data/regression/sters/comparisons/panel_GMFD_old_vs_new_var_summary.csv
  - /user/ab5405/summeraliaclimate/code/energy_uncertainty/data/regression/sters/comparisons/panel_GMFD_old_vs_new_worst20_by_absdiff.csv
  - /user/ab5405/summeraliaclimate/code/energy_uncertainty/data/regression/sters/comparisons/panel_GMFD_old_vs_new_precip_hdd_cdd_focus.csv


In [5]:
import pandas as pd
p = "/user/ab5405/summeraliaclimate/code/energy_uncertainty/data/regression/sters/comparisons/panel_GMFD_old_vs_new_precip_hdd_cdd_focus.csv"
df = pd.read_csv(p)
# try to find precip columns heuristically
cand = [c for c in df.columns if "precip" in c.lower() or "hdd" in c.lower() or "cdd" in c.lower()]
print("cols:", cand[:50])
print(df.head(20).to_string(index=False))



cols: []
                        col  n_both     old_mean     new_mean       old_sd       new_sd     corr    mean_diff  mean_abs_diff  mean_abs_rel_diff_pct  p50_abs_diff  p50_abs_rel_diff_pct  p90_abs_diff  p90_abs_rel_diff_pct  p95_abs_diff  p95_abs_rel_diff_pct  p99_abs_diff  p99_abs_rel_diff_pct
               precip2_GMFD    8301 4.562756e+04 4.383459e+04 5.550300e+04 5.543822e+04 0.986417 -1037.837610    2518.680427               6.133110    827.308938              3.185609   4669.455472             14.106491   7880.448585             23.073874  28264.061089             46.813578
                 cdd20_GMFD    8301 1.330697e+03 1.144008e+03 9.463452e+02 9.502361e+02 0.988222  -184.657844     192.958841              28.871370    163.427273             20.193415    376.304995             73.264852    492.863170             87.287440    595.376024             99.982343
            cdd20_TINV_GMFD    8301 1.325453e+03 1.141148e+03 9.409367e+02 9.455971e+02 0.988647  -160.044386     1

In [7]:
sed -n '1,60p' /user/ab5405/summeraliaclimate/code/energy_uncertainty/data/regression/sters/comparisons/panel_GMFD_old_vs_new_worst20_by_absdiff.csv


SyntaxError: invalid syntax (3414062803.py, line 1)

In [2]:
import pandas as pd

path = "/user/ab5405/summeraliaclimate/code/energy_uncertainty/data/country_climate/GMFD/GMFD_country_year_1971_2010.csv"
df = pd.read_csv(path)

df.head()
df.columns


Index(['iso', 'year', 'temp1_GMFD', 'temp2_GMFD', 'temp3_GMFD', 'temp4_GMFD',
       'hdd20_GMFD', 'cdd20_GMFD', 'polyAbove1_GMFD', 'polyAbove2_GMFD',
       'polyAbove3_GMFD', 'polyAbove4_GMFD', 'polyBelow1_GMFD',
       'polyBelow2_GMFD', 'polyBelow3_GMFD', 'polyBelow4_GMFD', 'precip1_GMFD',
       'precip2_GMFD', 'hdd20_TINV_GMFD', 'cdd20_TINV_GMFD'],
      dtype='object')

In [3]:
df[["precip1_GMFD", "precip2_GMFD"]].describe()


,precip1_GMFD,precip2_GMFD
count,7410.000000,7.410000e+03
mean,1264.312810,5.411713e+04
std,929.308171,1.146039e+05
min,7.378086,1.598790e+01
25%,597.539381,1.220577e+04
50%,1045.774369,3.125218e+04
75%,1749.429552,6.261719e+04
max,10164.871777,3.628418e+06


In [21]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

REGION_TEST = 101        # <-- pick an existing region_i
YEAR_TEST = 1995

OLD_PANEL = Path(
    "/user/ab5405/summeraliaclimate/code/energy_consumption/"
    "energy_data_release_2021oct21/DATA/regression/GMFD_TINV_clim_regsort.dta"
)

PR_PATH = Path(
    "/shared/share_hle/data/climate_raw/GMFD/pr_Amon_GMFD_historical_reanalysis_19810131-20101231.zarr"
)


In [19]:
old = pd.read_stata(OLD_PANEL, convert_categoricals=False)

row = old.query("region_i == @REGION_TEST and year == @YEAR_TEST")
if row.empty:
    raise ValueError("region_i / year not found in OLD panel")

old_p1 = float(row["precip1_GMFD"].iloc[0])
old_p2 = float(row["precip2_GMFD"].iloc[0])

print("OLD PANEL VALUES")
print(f"  precip1_GMFD = {old_p1:,.3f}")
print(f"  precip2_GMFD = {old_p2:,.3f}")


OLD PANEL VALUES
  precip1_GMFD = 1,543.166
  precip2_GMFD = 62,762.471


In [22]:
ds = xr.open_zarr(PR_PATH)
pr = ds["pr"].sel(time=str(YEAR_TEST))

# kg m-2 s-1 → mm/month
days = pr["time"].dt.days_in_month
pr_mm_month = pr * 86400.0 * days


In [23]:
# Load region mask (same one used in panel construction)
REGION_MASK = Path(
    "/shared/share_hle/data/1_estimation/3_regions/energy_regions.nc"
)

rm = xr.open_dataset(REGION_MASK)["region_i"]

mask = rm == REGION_TEST
pr_region = pr_mm_month.where(mask).mean(dim=["lat", "lon"], skipna=True)


FileNotFoundError: [Errno 2] No such file or directory: '/shared/share_hle/data/1_estimation/3_regions/energy_regions.nc'